# 🍜 SG/MY Food Recognition Demo

This notebook is a **thin demo layer** that uses the `sgmy_food` package.

**Sections:**
1. Installation & Setup
2. Quick Inference Demo
3. Push Base Model to Hub
4. Generate Training Dataset
5. Fine-tune with LoRA
6. Push Fine-tuned Model

> **Runtime:** GPU recommended. Go to `Runtime → Change runtime type → T4 GPU`

## 1️⃣ Installation & Setup

In [ ]:
# Install the package from GitHub
!pip install -q git+https://github.com/ghtliew18/sg-my-food-recognition.git#egg=sgmy-food-recognition[all]

print("✅ Package installed!")

In [ ]:
# Verify installation
import sgmy_food
print(f"Version: {sgmy_food.__version__}")
print(f"Supported foods: {len(sgmy_food.SG_MY_FOOD_LABELS)}")

## 2️⃣ Quick Inference Demo

In [ ]:
from sgmy_food import FoodRecognizer
from sgmy_food.model import display_results

# Load model (4-bit quantization for memory efficiency)
recognizer = FoodRecognizer(
    model_id="Qwen/Qwen2.5-VL-7B-Instruct",
    load_in_4bit=True,
).load()

In [ ]:
# Test with sample images
TEST_IMAGES = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a9/Nasi_lemak_on_banana_leaf.jpg/640px-Nasi_lemak_on_banana_leaf.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/6/65/Katong_laksa.jpg/640px-Katong_laksa.jpg",
]

for url in TEST_IMAGES:
    result = recognizer.recognize(url)
    display_results(url, result)
    print("\n")

In [ ]:
# Upload your own image
from google.colab import files

print("Upload a food image:")
uploaded = files.upload()

for filename in uploaded.keys():
    result = recognizer.recognize(filename)
    display_results(filename, result)

## 3️⃣ Push Base Model to Hub

Push the base model to your Hugging Face account **without any fine-tuning**.
This lets you start using your model immediately while you work on fine-tuning.

In [ ]:
from sgmy_food import HubManager

# === EDIT THESE VALUES ===
HF_USERNAME = "hong85"  # Your Hugging Face username
MODEL_NAME = "hong-sgmy-food-scanner"  # Your model name

# Login to Hugging Face
hub = HubManager(HF_USERNAME)
hub.login()

In [ ]:
# Push base model (this takes ~10-30 minutes)
repo_id = hub.push_base_model(
    model_name=MODEL_NAME,
    description="Singapore & Malaysian food recognition model based on Qwen2.5-VL-7B",
    private=False,
)

print(f"\n🎉 Model available at: https://huggingface.co/{repo_id}")

## 4️⃣ Generate Training Dataset

Generate a dataset of food images for fine-tuning.

In [ ]:
from sgmy_food.dataset import DatasetGenerator

# Generate dataset
generator = DatasetGenerator(
    output_dir="/content/sg_my_food_dataset",
    urls_per_term=30,
    image_size=512,
)

# Run full pipeline (URLs → Download → Annotations)
annotations = generator.run()

In [ ]:
# Visualize sample images
import matplotlib.pyplot as plt
from PIL import Image
import random

samples = random.sample(annotations, min(12, len(annotations)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, sample in zip(axes.flatten(), samples):
    try:
        img = Image.open(sample["image_path"])
        ax.imshow(img)
        ax.set_title(sample["label"], fontsize=10)
        ax.axis("off")
    except:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# (Optional) Push dataset to Hub
hub.push_dataset(
    dataset_name="sgmy-food-dataset",
    dataset_path="/content/sg_my_food_dataset",
)

## 5️⃣ Fine-tune with LoRA

In [ ]:
from sgmy_food.training import SgMyFoodDataset, LoRATrainer, load_annotations

# Load annotations
annotations = load_annotations("/content/sg_my_food_dataset/annotations_training.json")
print(f"Loaded {len(annotations)} training samples")

# Create dataset
dataset = SgMyFoodDataset(annotations, recognizer.processor)

In [ ]:
# Create trainer
trainer = LoRATrainer(
    model=recognizer.model,
    processor=recognizer.processor,
    lora_config={
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    training_config={
        "output_dir": "./sg_my_food_qlora",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2e-4,
    },
)

# Prepare model
trainer.prepare_model()

In [ ]:
# Run training
trainer.train(dataset)

In [ ]:
# Save adapter
ADAPTER_DIR = "./sg_my_food_lora_adapter"
trainer.save_adapter(ADAPTER_DIR)

## 6️⃣ Push Fine-tuned Model

In [ ]:
# Option 1: Push LoRA adapter only (~50MB)
hub.push_adapter(
    model_name=MODEL_NAME,
    adapter_path=ADAPTER_DIR,
)

In [ ]:
# Option 2: Push merged model (~14GB)
hub.push_merged_model(
    model_name=f"{MODEL_NAME}-merged",
    adapter_path=ADAPTER_DIR,
)

## 7️⃣ Test Fine-tuned Model

In [ ]:
# Load fine-tuned model from Hub
from sgmy_food import FoodRecognizer

ft_recognizer = FoodRecognizer(
    model_id="Qwen/Qwen2.5-VL-7B-Instruct",
    adapter_path=ADAPTER_DIR,  # or from Hub: f"{HF_USERNAME}/{MODEL_NAME}-lora"
).load()

# Test
result = ft_recognizer.recognize(TEST_IMAGES[0])
display_results(TEST_IMAGES[0], result)

---

## 📌 Summary

| Action | Code |
|--------|------|
| **Inference** | `FoodRecognizer().load().recognize(image)` |
| **Push base model** | `HubManager(user).push_base_model(name)` |
| **Generate dataset** | `DatasetGenerator(dir).run()` |
| **Fine-tune** | `LoRATrainer(model, processor).train(dataset)` |
| **Push adapter** | `HubManager(user).push_adapter(name, path)` |
| **Push merged** | `HubManager(user).push_merged_model(name, path)` |

For more details, see the [GitHub repository](https://github.com/ghtliew18/sg-my-food-recognition).